In [33]:
import tensorflow as tf
import numpy as np
from matplotlib import pyplot as plt
from collections import Counter

rng = np.random.default_rng(seed=42)

In [34]:
from tensorflow.keras.datasets import mnist
(train_images, _), (test_images, __) = mnist.load_data()

train_images = train_images[::3,:,:]

train_images = train_images / 256
test_images = test_images / 256

print(train_images.shape, test_images.shape)

(20000, 28, 28) (10000, 28, 28)


In [37]:
# 1 - all eigenvalues in the unit disc
# 0 - there exists at least 1 eigenvalue outside the disc r = 5

r = 5

y_train = []
for image in train_images:
  evs = np.abs(np.linalg.eigvals(image))
  y_train.append(1 if np.all(evs <= r) else 0)
y_train = np.array(y_train)

y_test = []
for image in test_images:
  evs = np.abs(np.linalg.eigvals(image))
  y_test.append(1 if np.all(evs <= r) else 0)
y_test = np.array(y_test)



In [38]:
y_train_counts = Counter(y_train)
y_test_counts = Counter(y_test)

print("y_train counts:", y_train_counts)
print("y_test counts:", y_test_counts)

y_train counts: Counter({np.int64(1): 10159, np.int64(0): 9841})
y_test counts: Counter({np.int64(0): 5169, np.int64(1): 4831})


In [39]:
shape = train_images[0].shape

model = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=shape),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(2)
])

model.compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])

model.fit(train_images, y_train, epochs=10)

/usr/local/lib/python3.11/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.8286 - loss: 0.3567
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.9241 - loss: 0.1698
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9442 - loss: 0.1339
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9561 - loss: 0.1075
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9665 - loss: 0.0836
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.9715 - loss: 0.0724
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9747 - loss: 0.0659
Epoch 8/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9776 - loss: 0.0542
Epoch 9/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9772 - loss: 0.0560
Epoch 10/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9862 - loss: 0.0398


In [40]:
y_predicted = model.predict(test_images)
y_predicted = np.argmax(y_predicted, axis=1)

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step


In [41]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_predicted)
print(cm)

[[4857  312]
 [ 410 4421]]


In [42]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_predicted))

              precision    recall  f1-score   support

           0       0.92      0.94      0.93      5169
           1       0.93      0.92      0.92      4831

    accuracy                           0.93     10000
   macro avg       0.93      0.93      0.93     10000
weighted avg       0.93      0.93      0.93     10000

